In [1]:
from pathlib import Path

from typing import Any

from bob.core import (
    System,
    Zone,
    Node,
    s223,
    Segment,
    Junction,
    bind_namespace,
    quantitykind,
    enum,
    turtle,
    get_datagraph,
    bind_model_namespace,
    dump,
)

from bob.devices.hvac.damper import ElectricalActuatedDamper
from bob.devices.hvac.coil import ChilledWaterCoil
from bob.devices.hvac.fan import Fan
from bob.systems.hvac.airhandlingunit import AirHandlingUnit
from bob.sensor.temperature import AirTemperatureSensor

from bob.space.physical import Building, Floor, Roof, Office
from bob.space.hvac import HVACSpace, HVACZone

from bob.connections.air import *


from bob.role import (
    Exhaust,
    Supply,
)
from bob.signal import (
    AnalogOut,
    AnalogIn,
)
from rdflib import Namespace, URIRef, BNode, Literal, RDF, RDFS, XSD

# from header import g36_header

# model_name = Path(__file__).stem
model_name = "B59"
__namespace__ = ex = bind_model_namespace("ex", f"urn:ex/{model_name}/")


def test_create_rooftop():
    config = {
        "params": {"label": "RTU-1", "comment": "Rooftop Unit"},
        "sensors": {
            ("DA-T", AirTemperatureSensor): {
                "comment": "Supply Air Temperature sensor"
            },
            ("RA-T", AirTemperatureSensor): {
                "comment": "Return Air Temperature sensor"
            },
            ("ZN-T", AirTemperatureSensor): {"comment": "Zone Air Temperature sensor"},
        },
        "contains": {
            ("SF-1", Fan): {"comment": "Supply Fan"},
            ("RF-1", Fan): {"comment": "Return Fan"},
            ("OAD-1", ElectricalActuatedDamper): {"comment": "Outside Air Damper"},
            ("RAD-1", ElectricalActuatedDamper): {"comment": "Return Air Damper"},
            ("CWC-1", ChilledWaterCoil): {"comment": "Chilled Water coil"},
        },
    }
    mixedAir = AirConnection(
        label="MIXED-AIR", comment="Where return air and outside air mix"
    )
    # rtu is a System
    rtu = AirHandlingUnit(config=config)

    # Relationships between devices
    rtu["OAD-1"] >> mixedAir
    rtu["RF-1"] >> mixedAir
    mixedAir >> rtu["SF-1"]
    rtu["SF-1"] >> rtu["CWC-1"]

    # Mapping of the system
    rtu.outsideAirInlet.mapsTo = rtu["OAD-1"].airInlet
    rtu.returnAirInlet.mapsTo = rtu["RF-1"].airInlet
    rtu.supplyAirOutlet.mapsTo = rtu["CWC-1"].airOutlet

    return_plenum = AirConnection(
        label="Return Air Plenum", comment="Air returns from zone here"
    )
    return_plenum >> rtu["RF-1"].airInlet
    supply_duct = AirConnection(
        label="Supply Air Duct", comment="Air returns from zone here"
    )
    rtu["CWC-1"].airOutlet >> supply_duct
    rtu["DA-T"].hasMeasurementLocation = supply_duct
    rtu["RA-T"].hasMeasurementLocation = rtu["RF-1"].airOutlet

    bldg = Building(label="B59 Building")
    roof = Roof(label="Roof of building")
    floor1 = Floor(label="One big floor which is a common space")
    office1 = Office(label="Director Office")
    floor1_hvacspace = HVACSpace(label="HVAC Space for floor 1")
    rtu_zone = HVACZone(label="Common workspace zone for HVAC")

    bldg > floor1
    floor1_hvacspace < floor1

    supply_duct >> floor1_hvacspace.airInlet
    floor1_hvacspace.airOutlet >> return_plenum

    rtu_zone > floor1_hvacspace
    rtu_zone.airInlet.mapsTo = supply_duct
    rtu_zone.airOutlet.mapsTo = return_plenum

    rtu.hasPhysicalLocation = roof
    rtu["ZN-T"].hasMeasurementLocation = floor1_hvacspace.airOutlet
    rtu["ZN-T"].hasPhysicalLocation = office1
    return rtu

In [2]:
rtu = test_create_rooftop()

ARGS :  ()
ARGS :  ()
ARGS :  ()


In [3]:
rtu['DA-T']

{'node': rdflib.term.URIRef('urn:ex/B59/00002'), 'label': 'DA-T', 'comment': 'Supply Air Temperature sensor', 'hasRole': None, 'hasPhysicalLocation': None, 'hasMeasurementLocation': {'node': rdflib.term.URIRef('urn:ex/B59/00041'), 'label': 'Supply Air Duct', 'comment': 'Air returns from zone here', 'hasMedium': rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Medium-Air')}, 'hasMeasurementPrecision': None, 'hasMeasurementUncertainty': None, 'hasMaxRange': None, 'hasMinRange': None, 'hasMedium': rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Medium-Air'), 'measuresSubstance': rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Medium-Air'), 'observesProperty': {'node': rdflib.term.URIRef('urn:ex/B59/00003'), 'label': 'DA-T.Measure', 'comment': '', 'hasQuantityKind': rdflib.term.URIRef('http://qudt.org/vocab/quantitykind/Temperature'), 'unit': rdflib.term.URIRef('http://qudt.org/vocab/unit/DEG_C'), 'hasVal

In [4]:
from bob.systems.functionblock import FunctionBlock, AnalogIn

In [11]:
class MyFB(FunctionBlock):
    ai1: SystemConnectionPoint


fb = MyFB(label='Read DA-T', comment="Convert to degF")
fb.ai1.mapsTo = rtu['DA-T']

In [12]:
fb

{'node': rdflib.term.URIRef('urn:ex/B59/00057'), 'label': 'Read DA-T', 'comment': 'Convert to degF', 'hasPhysicalLocation': None, 'hasDomain': None, 'ai1': {'node': rdflib.term.URIRef('urn:ex/B59/00058'), 'label': 'Read DA-T.ai1', 'comment': '', 'hasMedium': None, 'hasDirection': None, 'connectsThrough': None, 'isSystemConnectionPointOf': None, 'mapsTo': {'node': rdflib.term.URIRef('urn:ex/B59/00002'), 'label': 'DA-T', 'comment': 'Supply Air Temperature sensor', 'hasRole': None, 'hasPhysicalLocation': None, 'hasMeasurementLocation': {'node': rdflib.term.URIRef('urn:ex/B59/00041'), 'label': 'Supply Air Duct', 'comment': 'Air returns from zone here', 'hasMedium': rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Medium-Air')}, 'hasMeasurementPrecision': None, 'hasMeasurementUncertainty': None, 'hasMaxRange': None, 'hasMinRange': None, 'hasMedium': rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Medium-Air'), 'measuresSubstance': rdf

In [10]:
fb

{'node': rdflib.term.URIRef('urn:ex/B59/00055'), 'label': 'Read DA-T', 'comment': 'Convert to degF', 'hasPhysicalLocation': None, 'hasDomain': None, '_system_connection_points': {}}